## DATA PREPROCESSING

In [2]:
import numpy as np
import pandas as pd
import ast
import nltk
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [467]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [469]:
movies.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [5]:
credits.head(2)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [7]:
movies.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

In [471]:
movies = movies.merge(credits, on='title')

In [473]:
movies = movies[['id','title','genres','keywords','overview','popularity','production_companies','cast','crew','release_date']]

In [475]:
popularity_tag = []
for val in movies['popularity']:
    if val > 100:
        popularity_tag.append(['popularity_high'])
    elif val >= 50:
        popularity_tag.append(['popularity_med'])
    else:
        popularity_tag.append(['popularity_low'])
movies['popularity_tag'] = popularity_tag

In [477]:
movies['description'] = movies['overview']

In [487]:
movies.isnull().sum()

id                      0
title                   0
genres                  0
keywords                0
overview                0
popularity              0
production_companies    0
cast                    0
crew                    0
release_date            0
popularity_tag          0
description             0
dtype: int64

In [489]:
movies.dropna(inplace=True)

In [491]:
def convert(arr):
    arr = ast.literal_eval(arr)
    genre = []
    for x in arr:
        genre.append(x["name"])
    return genre
    
def convertCast(arr):
    arr = ast.literal_eval(arr)
    cast = []
    for i in range(min(3,len(arr))):
        cast.append(arr[i]["name"])
    return cast

def convertCrew(arr):
    arr = ast.literal_eval(arr)
    crew = []
    jobs = ['Director', 'Writer', 'Producer', 'Screenplay']
    for person in arr:
        if person["job"] in jobs:
            crew.append(person["name"])
    crew = list(set(crew))
    return crew

In [493]:
movies["genres"] = movies["genres"].apply(convert)
movies["keywords"] = movies["keywords"].apply(convert)
movies["production_companies"] = movies["production_companies"].apply(convert)
movies["cast"] = movies["cast"].apply(convertCast)
movies["crew"] = movies["crew"].apply(convertCrew)

In [497]:
def joinWords(arr):
    for i in range(len(arr)):
        arr[i] = "".join(arr[i].split(' '))
    return arr

In [499]:
movies["production_companies"] = movies["production_companies"].apply(joinWords)
movies["cast"] = movies["cast"].apply(joinWords)
movies["crew"] = movies["crew"].apply(joinWords)
movies["keywords"] = movies["keywords"].apply(joinWords)

In [501]:
movies["overview"] = movies["overview"].apply(lambda x:x.split())

In [503]:
movies['tags'] = movies['genres'] + movies['keywords'] + movies['overview'] + movies['production_companies'] + movies['cast'] + movies['crew'] + movies['popularity_tag']

In [505]:
movies.head()

,id,title,genres,keywords,overview,popularity,production_companies,cast,crew,release_date,popularity_tag,description,tags
0,19995,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[cultureclash, future, spacewar, spacecolony, ...","[In, the, 22nd, century,, a, paraplegic, Marin...",150.437577,"[IngeniousFilmPartners, TwentiethCenturyFoxFil...","[SamWorthington, ZoeSaldana, SigourneyWeaver]","[JamesCameron, JonLandau]",2009-12-10,[popularity_high],"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction, ..."
1,285,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[Captain, Barbossa,, long, believed, to, be, d...",139.082615,"[WaltDisneyPictures, JerryBruckheimerFilms, Se...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]","[GoreVerbinski, TerryRossio, PatSandston, Pete...",2007-05-19,[popularity_high],"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action, ocean, drugabuse,..."
2,206647,Spectre,"[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[A, cryptic, message, from, Bond’s, past, send...",107.376788,"[ColumbiaPictures, Danjaq, B24]","[DanielCraig, ChristophWaltz, LéaSeydoux]","[BarbaraBroccoli, NealPurvis, MichaelG.Wilson,...",2015-10-26,[popularity_high],A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime, spy, basedonnovel, ..."
3,49026,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[Following, the, death, of, District, Attorney...",112.312950,"[LegendaryPictures, WarnerBros., DCEntertainme...","[ChristianBale, MichaelCaine, GaryOldman]","[JonathanNolan, EmmaThomas, CharlesRoven, Chri...",2012-07-16,[popularity_high],Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller, dccomics, cri..."
4,49529,John Carter,"[Action, Adventure, Science Fiction]","[basedonnovel, mars, medallion, spacetravel, p...","[John, Carter, is, a, war-weary,, former, mili...",43.926995,[WaltDisneyPictures],"[TaylorKitsch, LynnCollins, SamanthaMorton]","[LindseyCollins, JimMorris, ColinWilson, Andre...",2012-03-07,[popularity_low],"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction, basedonno..."


In [507]:
new_df = movies[['id','title','tags','popularity','description','release_date']]

In [509]:
new_df.loc[:, 'tags'] = new_df['tags'].apply(lambda x: " ".join(x).lower())
new_df.loc[:, 'title'] = new_df['title'].apply(lambda x: x.lower())

In [511]:
new_df.head()

,id,title,tags,popularity,description,release_date
0,19995,avatar,action adventure fantasy science fiction cultu...,150.437577,"In the 22nd century, a paraplegic Marine is di...",2009-12-10
1,285,pirates of the caribbean: at world's end,adventure fantasy action ocean drugabuse exoti...,139.082615,"Captain Barbossa, long believed to be dead, ha...",2007-05-19
2,206647,spectre,action adventure crime spy basedonnovel secret...,107.376788,A cryptic message from Bond’s past sends him o...,2015-10-26
3,49026,the dark knight rises,action crime drama thriller dccomics crimefigh...,112.312950,Following the death of District Attorney Harve...,2012-07-16
4,49529,john carter,action adventure science fiction basedonnovel ...,43.926995,"John Carter is a war-weary, former military ca...",2012-03-07


## VECTORIZATION (BoW)

In [513]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [515]:
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [447]:
new_df.loc[:,'tags'] = new_df['tags'].apply(stem)

In [448]:
cv = CountVectorizer(max_features=5000,stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()

In [451]:
similarity = cosine_similarity(vectors)

In [453]:
def recommend(movie):
    movie = movie.lower()
    movie_index = new_df[new_df['title'] == movie.lower()].index[0]
    alikes = similarity[movie_index]
    movie_list = sorted(list(enumerate(alikes)), reverse=True, key = lambda x:x[1])[1:6]
    movie_names = []
    for i in movie_list:
        movie_names.append(new_df.iloc[i[0]].title) 
    return movie_names

In [455]:
recommend("The vatican tapes")

['the helix... loaded',
 'independence daysaster',
 'slither',
 'impostor',
 'under the skin']

In [517]:
pickle.dump(new_df, open("data.pkl", "wb"))